In [1]:
%cd ..

C:\Users\malte\EvoMUSART23-repro-code


In [2]:
from evoaudio.sample_library import SampleLibrary
sample_lib = SampleLibrary()

Loading samples:   0%|          | 0/6826 [00:00<?, ?it/s]

In [3]:
import os
import pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from evoaudio.sample_library import SampleLibrary
from evoaudio.chord_metrics import chord_metrics
from evoaudio.instrument_metric import avg_hausdorff_instruments, hausdorff_instruments_full_piece_GD_IGD
from evoaudio.jaccard import jaccard_error_GD_IGD
from evoaudio.jaccard import class_mode

In [12]:
run_id = 0 
obj_name = "stft_kl_divergence__stft_js_divergence__stft_cosh__iirt_cosh__cqt_kl_divergence"

base_path = "experiments/moea/tiny_aam/smsemoa/recall"

# csv_path = os.path.join(base_path, f"{run_id}__{obj_name}.csv")
# df = pd.read_csv(csv_path)

# df.head()

In [6]:
def remove_empty_onsets(annotations: dict[int, list[tuple]]):
    return {
        t: ann
        for t, ann in annotations.items()
        if len(ann) > 0
    }

In [13]:
all_files = [
    f for f in os.listdir(base_path)
    if f.endswith("_300_populations.pkl")
]

print(f"Found {len(all_files)} population files.")

rows = []

# -------------------------------------------------
# Process each file
# -------------------------------------------------

for file in all_files:
    print("analyzing file:", file)
    full_path = os.path.join(base_path, file)
    run_id, rest = file.split("__", 1)
    obj_string = rest.replace("_300_populations.pkl", "")
    run_id = int(run_id)
    
    with open(full_path, "rb") as f:
        result_logs = pickle.load(f)
    
    for name, annotation_dict, onsets, target_sr, pop in result_logs:
        
        pop._expand(sample_lib)
        filtered_annotations = remove_empty_onsets(annotation_dict)
        
        GD, IGD = jaccard_error_GD_IGD(
            pop,
            filtered_annotations,
            archive = False
        )
        
        rows.append({
            "run_id": run_id,
            "objectives": obj_string,
            "piece": name,
            "GD": GD,
            "IGD": IGD
        })

# -------------------------------------------------
# Final DataFrame
# -------------------------------------------------

df = pd.DataFrame(rows)

print(df.head())
print("\nShape:", df.shape)

df

Found 90 population files.
analyzing file: 0__stft_kl_divergence__stft_js_divergence_300_populations.pkl
analyzing file: 0__stft_kl_divergence__stft_js_divergence__stft_cosh_300_populations.pkl
analyzing file: 0__stft_kl_divergence__stft_js_divergence__stft_cosh__iirt_cosh_300_populations.pkl
analyzing file: 0__stft_kl_divergence__stft_js_divergence__stft_cosh__iirt_cosh__cqt_kl_divergence_300_populations.pkl
analyzing file: 0__stft_kl_divergence__stft_js_divergence__stft_cosh__iirt_cosh__cqt_kl_divergence__melspectrogram_cosh_300_populations.pkl
analyzing file: 0__stft_kl_divergence__stft_js_divergence__stft_cosh__iirt_cosh__cqt_kl_divergence__melspectrogram_cosh__hybrid_cqt_cosh_300_populations.pkl
analyzing file: 0__stft_kl_divergence__stft_js_divergence__stft_cosh__iirt_cosh__cqt_kl_divergence__melspectrogram_cosh__hybrid_cqt_cosh__cqt_kl_divergence_300_populations.pkl
analyzing file: 0__stft_kl_divergence__stft_js_divergence__stft_cosh__iirt_cosh__cqt_kl_divergence__melspectrogram

analyzing file: 6__stft_kl_divergence__stft_js_divergence__stft_cosh__iirt_cosh__cqt_kl_divergence__melspectrogram_cosh__hybrid_cqt_cosh_300_populations.pkl
analyzing file: 6__stft_kl_divergence__stft_js_divergence__stft_cosh__iirt_cosh__cqt_kl_divergence__melspectrogram_cosh__hybrid_cqt_cosh__cqt_kl_divergence_300_populations.pkl
analyzing file: 6__stft_kl_divergence__stft_js_divergence__stft_cosh__iirt_cosh__cqt_kl_divergence__melspectrogram_cosh__hybrid_cqt_cosh__cqt_kl_divergence__stft_beta_divergence_300_populations.pkl
analyzing file: 6__stft_kl_divergence__stft_js_divergence__stft_cosh__iirt_cosh__cqt_kl_divergence__melspectrogram_cosh__hybrid_cqt_cosh__cqt_kl_divergence__stft_beta_divergence__cqt_beta_divergence_300_populations.pkl
analyzing file: 7__stft_kl_divergence__stft_js_divergence_300_populations.pkl
analyzing file: 7__stft_kl_divergence__stft_js_divergence__stft_cosh_300_populations.pkl
analyzing file: 7__stft_kl_divergence__stft_js_divergence__stft_cosh__iirt_cosh_300

,run_id,objectives,piece,GD,IGD
0,0,stft_kl_divergence__stft_js_divergence,0001,0.929770,0.792126
1,0,stft_kl_divergence__stft_js_divergence,3000,0.976877,0.867956
2,0,stft_kl_divergence__stft_js_divergence,2462,0.987626,0.930191
3,0,stft_kl_divergence__stft_js_divergence,2828,0.958484,0.800994
4,0,stft_kl_divergence__stft_js_divergence,0192,0.888012,0.755527
...,...,...,...,...,...
1795,9,stft_kl_divergence__stft_js_divergence__stft_c...,2395,0.867097,0.897984
1796,9,stft_kl_divergence__stft_js_divergence__stft_c...,2602,0.894558,0.893170
1797,9,stft_kl_divergence__stft_js_divergence__stft_c...,2990,0.850082,0.850433
1798,9,stft_kl_divergence__stft_js_divergence__stft_c...,0620,0.970090,0.899307


In [14]:
df.to_csv(os.path.join(base_path, "jaccard_GD_IGD_SMSEMOA_300.csv"), index=False)

In [12]:
from evoaudio.individual import BaseIndividual
from evoaudio.population import Population
from evoaudio.population import ArchiveRecord
from evoaudio.pitch import Pitch
from evoaudio.pitch import DrumHit

In [13]:
def to_pitch(p):
    """
    Converts raw annotation pitch into Pitch or DrumHit enum.
    Handles:
      - "+41", "41"
      - "c4", "fis3"
      - existing Pitch / DrumHit
    """

    # already correct type
    if isinstance(p, (Pitch, DrumHit)):
        return p

    # numeric string like "+41"
    if isinstance(p, str) and p.lstrip("+").isdigit():
        return Pitch(int(p.lstrip("+")))

    # symbolic pitch like "c4"
    if isinstance(p, str):
        p_clean = p.lower().strip()

        # try Pitch enum name
        if hasattr(Pitch, p_clean):
            return getattr(Pitch, p_clean)

        # try DrumHit enum name
        if hasattr(DrumHit, p_clean):
            return getattr(DrumHit, p_clean)

    raise ValueError(f"Unknown pitch format: {p}")

In [14]:
def build_feasible_optimal_population(filtered_annotations, sample_lib, phi=1.0):

    pop = Population()
    pop.individuals = []
    pop.archive = {}
    all_events = []

    for onset, events in filtered_annotations.items():

        individual = BaseIndividual(phi=phi)
        fitness_vec = []
        if events not in all_events:
            all_events.append(events)

            for instrument, pitch in events:

                pitch = to_pitch(pitch)

                try:
                    sample = sample_lib.get_sample(instrument, None, pitch)

                except:

                    fallback_inst, fallback_style = sample_lib.get_random_instrument_for_pitch(pitch)

                    sample = sample_lib.get_sample(
                        fallback_inst,
                        fallback_style,
                        pitch
                    )

                individual.samples.append(sample)
                fitness_vec.append([0.0])

            individual.fitness_per_onset = [fitness_vec]
            individual.fitness = np.array([0.0])
            individual.recalc_fitness = False

            pop.individuals.append(individual)

            pop.archive[onset] = ArchiveRecord(
                onset=onset,
                fitness_vec=np.array(fitness_vec),
                individual=individual
            )

    pop.sort_individuals_by_fitness()

    return pop

In [15]:
def population_stats(pop):
    n_individuals = len(pop.individuals)

    n_samples = [len(ind.samples) for ind in pop.individuals]

    return {
        "n_individuals": n_individuals,
        "total_samples": sum(n_samples),
        "mean_samples_per_individual": np.mean(n_samples),
        "min_samples": np.min(n_samples),
        "max_samples": np.max(n_samples),
    }

In [16]:
optimal_rows = []
rows = []

all_files = [
    f for f in os.listdir(base_path)
    if f.endswith("_300_populations.pkl")
]
run_id, rest = all_files[0].split("__", 1)
obj_string = rest.replace("_300_populations.pkl", "")
run_id = int(run_id)
full_path = os.path.join(base_path, all_files[0])
with open(full_path, "rb") as f:
    result_logs = pickle.load(f)

for name, annotation_dict, onsets, target_sr, pop in result_logs:

    filtered_annotations = remove_empty_onsets(annotation_dict)

    optimal_pop = build_feasible_optimal_population(
        filtered_annotations,
        sample_lib,
        phi=1.0
    )

    stats = population_stats(optimal_pop)

    optimal_rows.append({
        "piece": name,
        "n_onsets": len(filtered_annotations),
        "n_individuals": stats["n_individuals"],
        "total_samples": stats["total_samples"],
        "mean_samples_per_individual": stats["mean_samples_per_individual"],
        "max_samples": stats["max_samples"],
    })
    
    GD, IGD = jaccard_error_GD_IGD(
            optimal_pop,
            filtered_annotations,
            archive = False
        )
    rows.append({
        "run_id": run_id,
        "objectives": obj_string,
        "piece": name,
        "GD": GD,
        "IGD": IGD
    })
    
df = pd.DataFrame(rows)

print(df.head(20))
print("\nShape:", df.shape)
optimal_rows

    run_id          objectives piece        GD       IGD
0        0  stft_js_divergence  0001  0.002778  0.004529
1        0  stft_js_divergence  3000  0.058924  0.043328
2        0  stft_js_divergence  2462  0.038056  0.054491
3        0  stft_js_divergence  2828  0.000000  0.000000
4        0  stft_js_divergence  0192  0.000000  0.000000
5        0  stft_js_divergence  1711  0.030014  0.009536
6        0  stft_js_divergence  2269  0.028521  0.023241
7        0  stft_js_divergence  0080  0.014642  0.013459
8        0  stft_js_divergence  1545  0.001701  0.002646
9        0  stft_js_divergence  0758  0.007035  0.009221
10       0  stft_js_divergence  1050  0.000000  0.000000
11       0  stft_js_divergence  0989  0.000000  0.000000
12       0  stft_js_divergence  1941  0.017457  0.007613
13       0  stft_js_divergence  2720  0.000000  0.000000
14       0  stft_js_divergence  2395  0.000000  0.000000
15       0  stft_js_divergence  1014  0.008833  0.012904
16       0  stft_js_divergence 

[{'piece': '0001',
  'n_onsets': 368,
  'n_individuals': 120,
  'total_samples': 390,
  'mean_samples_per_individual': 3.25,
  'max_samples': 4},
 {'piece': '3000',
  'n_onsets': 369,
  'n_individuals': 133,
  'total_samples': 795,
  'mean_samples_per_individual': 5.977443609022556,
  'max_samples': 8},
 {'piece': '2462',
  'n_onsets': 369,
  'n_individuals': 122,
  'total_samples': 671,
  'mean_samples_per_individual': 5.5,
  'max_samples': 7},
 {'piece': '2828',
  'n_onsets': 381,
  'n_individuals': 153,
  'total_samples': 887,
  'mean_samples_per_individual': 5.7973856209150325,
  'max_samples': 8},
 {'piece': '0192',
  'n_onsets': 392,
  'n_individuals': 62,
  'total_samples': 323,
  'mean_samples_per_individual': 5.209677419354839,
  'max_samples': 6},
 {'piece': '1711',
  'n_onsets': 464,
  'n_individuals': 126,
  'total_samples': 801,
  'mean_samples_per_individual': 6.357142857142857,
  'max_samples': 8},
 {'piece': '2269',
  'n_onsets': 468,
  'n_individuals': 299,
  'total_sa

In [15]:
df.describe()

,run_id,GD,IGD
count,20.0,20.000000,20.000000
mean,0.0,0.013827,0.011453
std,0.0,0.016092,0.014741
min,0.0,0.000000,0.000000
25%,0.0,0.000000,0.000000
50%,0.0,0.007934,0.008417
75%,0.0,0.023187,0.015104
max,0.0,0.058924,0.054491


In [11]:
library_instruments = sample_lib.get_all_instruments().keys()
print(library_instruments)

dict_keys(['AcousticGuitar', 'Balalaika', 'Bandura', 'BanjoFramus', 'Banjolin', 'Bass', 'Bassoon', 'Bawu', 'BouzoukiSakis', 'Cello', 'CeylonGuitar', 'Clarinet', 'Contrabassoon', 'CumbusOpenStrings', 'DallapeAccordion', 'Dilruba', 'Domra', 'Drums', 'DungDkarTrumpet', 'EgyptianFiddle', 'ElectricBass', 'ElectricGuitar', 'ElectricPiano', 'Erhu', 'Flute', 'Fujara', 'Horn', 'JinghuOperaviolin', 'KanteleLowWideRange', 'Melodica', 'MorinKhuurViolin', 'Oboe', 'Organ', 'Oud', 'Panflute', 'Piano', 'Pinkillo', 'PivanaFlute', 'Saxophone', 'ScaleChangerHarmonium', 'Shakuhachi', 'Sitar', 'TampuraPAndBrothers', 'Tanbur', 'Trombone', 'Trumpet', 'Tuba', 'TurkeySaz', 'Ukulele', 'Viola', 'Violin'])


In [14]:
all_files = [
    f for f in os.listdir(base_path)
    if f.endswith("_300_populations.pkl")
]

print(f"Found {len(all_files)} population files.")

rows = []
optimal_rows = []

for file in all_files:
    print("analyzing file:", file)
    full_path = os.path.join(base_path, file)
    
    run_id, rest = file.split("__", 1)
    obj_string = rest.replace("_300_populations.pkl", "")
    run_id = int(run_id)
    
    with open(full_path, "rb") as f:
        result_logs = pickle.load(f)
    
    for name, annotation_dict, onsets, target_sr, pop in result_logs:
        
        filtered_annotations = remove_empty_onsets(annotation_dict)
        
        # -------------------------------------------------
        # Build optimal population
        # -------------------------------------------------
        
        optimal_population = []
        onset_count = 0
        
        for onset_time, events in filtered_annotations.items():
            onset_count += 1
            
            individual = []
            
            for instrument, pitch in events:
                
                # Case 1: instrument exists
                if instrument in library_instruments:
                    individual.append((instrument, pitch))
                
                # Case 2: instrument missing → replace
                else:
                    replacement = random.choice(library_instruments)
                    individual.append((replacement, pitch))    
                    
            optimal_population.append(individual)
        
        optimal_rows.append({
            "piece": name,
            "run_id": run_id,
            "objectives": obj_string,
            "n_onsets": onset_count,
            "optimal_population_size": len(optimal_population)
        })
    break

# -------------------------------------------------
# DataFrame
# -------------------------------------------------

df_optimal = pd.DataFrame(optimal_rows)

print(df_optimal.head())
print("\nShape:", df_optimal.shape)

Found 90 population files.
analyzing file: 0__stft_kl_divergence__stft_js_divergence_300_populations.pkl
  piece  run_id                              objectives  n_onsets  \
0  0001       0  stft_kl_divergence__stft_js_divergence       368   
1  3000       0  stft_kl_divergence__stft_js_divergence       369   
2  2462       0  stft_kl_divergence__stft_js_divergence       369   
3  2828       0  stft_kl_divergence__stft_js_divergence       381   
4  0192       0  stft_kl_divergence__stft_js_divergence       392   

   optimal_population_size  
0                      368  
1                      369  
2                      369  
3                      381  
4                      392  

Shape: (20, 5)


In [8]:
import re
import pandas as pd
from evoaudio.population import Population

results = []
BASE_FOLDER = "./experiments/tiny_aam/"
folder_pattern = re.compile(r"stft_js_divergence_(\d+)$") 

for folder in os.listdir(BASE_FOLDER):
    folder_path = os.path.join(BASE_FOLDER, folder)
    if not os.path.isdir(folder_path):
        continue

    match = folder_pattern.match(folder)
    if not match:
        print(f"⚠️ Skipping {folder} — doesn’t match expected pattern.")
        continue

    
    index = match.groups()
    base_name = "stft_js_diveregence"

    # Split base_name into feature + distance (split at first underscore)
    if "_" in base_name:
        feature, distance = base_name.split("_", 1)
    else:
        feature, distance = base_name, "unknown"

    print(f"📂 Analyzing: Feature={feature}, Distance={distance}, Piece={index}")

    # --- Process all files inside ---
    for filename in os.listdir(folder_path):
        if filename.endswith(".pkl") and not filename.endswith("_annotations.pkl"):
            name = filename.replace(".pkl", "")
            result_path = os.path.join(folder_path, f"{name}.pkl")
            annotation_path = os.path.join(folder_path, f"{name}_annotations.pkl")

            if not os.path.exists(annotation_path):
                print(f"⚠️ Missing annotations for {name}, skipping.")
                continue

            # --- Load population ---
            pop = Population.from_file(result_path, expand=True, sample_lib=sample_lib)

            # --- Load annotations ---
            with open(annotation_path, "rb") as f:
                annotation_data = pickle.load(f)
            annotations = annotation_data["annotations"]

            # --- Align and filter ---
            annotation_items = list(annotations.items())
            archive_items = list(pop.archive.items())
            min_len = min(len(annotation_items), len(archive_items))

            aligned_annotations = {
                i: annotation_items[i][1]
                for i in range(min_len)
                if len(annotation_items[i][1]) > 0
            }
            aligned_archive = {
                i: archive_items[i][1]
                for i in range(min_len)
                if len(annotation_items[i][1]) > 0
            }

            pop.archive = aligned_archive
            annotations = aligned_annotations
            
            #instrument_err = hausdorff_instruments_full_piece_GD_IGD(pop, annotations, algorithm="EA")
            jaccard_err = jaccard_error_GD_IGD(pop, annotations)
            
            results.append({
                "piece": index,
                "file": name,
                #"instrument_error": instrument_err,
                "Jaccard": jaccard_err
            })
            print("Done: ", filename)
            
df = pd.DataFrame(results)

⚠️ Skipping 300_1_10000_0.05_5_10_1_20_0.9954_15_10_1sec_convlog — doesn’t match expected pattern.
⚠️ Skipping 300_1_10000_0.05_5_10_1_20_0.9954_15_10_1sec_repeat — doesn’t match expected pattern.
⚠️ Skipping 300_1_10000_0.05_5_10_1_20_0.9954_15_20_1sec_preprocess — doesn’t match expected pattern.
⚠️ Skipping 300_1_10000_0.05_5_5_1_8_0.9954_15_20_1sec_preprocess — doesn’t match expected pattern.
⚠️ Skipping all_random_0 — doesn’t match expected pattern.
⚠️ Skipping all_random_1 — doesn’t match expected pattern.
⚠️ Skipping all_random_10 — doesn’t match expected pattern.
⚠️ Skipping all_random_11 — doesn’t match expected pattern.
⚠️ Skipping all_random_12 — doesn’t match expected pattern.
⚠️ Skipping all_random_13 — doesn’t match expected pattern.
⚠️ Skipping all_random_14 — doesn’t match expected pattern.
⚠️ Skipping all_random_15 — doesn’t match expected pattern.
⚠️ Skipping all_random_16 — doesn’t match expected pattern.
⚠️ Skipping all_random_17 — doesn’t match expected pattern.
⚠️ 

KeyboardInterrupt: 

In [10]:
df

,piece,file,Jaccard
0,"(0,)",3000,"(0.9788809523809525, 0.9224712279590329)"
1,"(1,)",2828,"(0.8510555555555556, 0.9179984130771532)"
2,"(10,)",2841,"(0.4783333333333332, 0.8757843137254903)"
3,"(11,)",0758,"(0.9664603174603176, 0.905141344599176)"
4,"(12,)",0192,"(0.9577380952380953, 0.8909489390994493)"
5,"(13,)",2395,"(0.7720158730158729, 0.941405433646813)"
6,"(14,)",2269,"(0.9635108225108227, 0.9578633557800226)"
7,"(15,)",1014,"(0.9813134920634919, 0.8779747031246282)"
8,"(16,)",0080,"(0.893, 0.8687894434882386)"
9,"(17,)",0989,"(0.8970378787878788, 0.8590250706031152)"


In [11]:
df.to_csv(os.path.join(BASE_FOLDER, "Jaccard_GD_IGD_stft_js_divergence_aam_final_population.csv"), index=False)